# Development Tutorial
## Getting Started
This tutorial focuses on selecting the development factors. 

Be sure to make sure your packages are updated. For more info on how to update your pakages, visit [Keeping Packages Updated](https://chainladder-python.readthedocs.io/en/latest/library/install.html#keeping-packages-updated).

In [0]:
# Black linter, optional
# import jupyter_black as jb
# jb.load()

import pandas as pd
import numpy as np
import chainladder as cl

print("pandas: " + pd.__version__)
print("numpy: " + np.__version__)
print("chainladder: " + cl.__version__)

## Disclaimer
Note that a lot of the examples shown might not be applicable in a real world scenario, and is only meant to demonstrate some of the functionalities included in the package. The user should always follow all applicable laws, the Code of Professional Conduct, applicable Actuarial Standards of Practice, and exercise their best actuarial judgement.

## Testing for Violation of Chain Ladder's Assumptions

The chain ladder method is based on the strong assumptions of independence across origin periods and across valuation periods. Mack developed tests to verify if these assumptions hold, and these tests have been implemented in the `chainladder` package.

Before the chain ladder model can be used, we should verify that the data satisfies the underlying assumptions using tests at the desired confidence interval level. If assumptions are violated, we should consider if ultimates can be estimated using other models.

There are two main tests that we need to perform:
- The `valuation_correlation` test: 
    - This test tests for the assumption of independence of accident years. In fact, it tests for correlation across calendar periods (diagonals), and by extension, origin periods (rows).
    - An additional parameter, `total`, can be passed, depending on if we want to calculate valuation correlation in total across all origins (`True`), or for each origin separately (`False`).
    - The test uses Z-statistic.
- The `development_correlation` test:
    - This test tests for the assumption of independence of the chain ladder method that assumes that subsequent development factors are not correlated (columns).
    - The test uses T-statistic.

In [0]:
raa = cl.load_sample("raa")
print(
    "Are valuation years correlated? Or, are the origins correlated?",
    raa.valuation_correlation(p_critical=0.1, total=True).z_critical.values,
)
print(
    "Are development periods coorelated?",
    raa.development_correlation(p_critical=0.5).t_critical.values,
)

The above tests show that the `raa` triangle is independent in both cases, suggesting that there is no evidence that the chain ladder model is not an appropriate method to develop the ultimate amounts. It is suggested to review Mack's papers to ensure a proper understanding of the methodology and the choice of `p_critical`.

Mack also demonstrated that we can test for valuation years' correlation. To test for each valuation year's correlation individually, we set `total` to `False`.

In [0]:
raa.valuation_correlation(p_critical=0.1, total=False).z_critical

Note that the tests are run on the entire 4 dimensions of the `triangle`.

## Estimator Basics

All development methods follow the `sklearn` estimator API.  These estimators have a few properties that are worth getting used to.

We instantiate the estimator with your choice of assumptions.  In the case where we don't opt for any assumptions, defaults are chosen for you.

At this point, we've chosen an estimator and assumptions (even if default) but we have not shown our estimator a `Triangle`.  At this point it is merely instructions on how to fit development patterns, but no patterns exist as of yet.

All estimators have a `fit` method and you can pass a triangle to your estimator.  Let's `fit` a `Triangle` in a `Development` estimator.  Let's also assign the estimator to a variable so we can reference attributes about it.

In [0]:
genins = cl.load_sample("genins")
dev = cl.Development().fit(genins)

Now that we have `fit` a `Development` estimator, it has many additional properties that didn't exist before fitting.  For example, 
we can view the `ldf_`

In [0]:
dev.ldf_

We can view the `cdf_`

In [0]:
dev.cdf_

We can also convert between LDFs and CDFs using incr_to_cum() and cum_to_incr() similar to triangles.

In [0]:
dev.ldf_.incr_to_cum()

In [0]:
dev.cdf_.cum_to_incr()

Notice these attributes have a trailing underscore (`_`). This is scikit-learn's API convention, as its [documentation](https://scikit-learn.org/dev/developers/develop.html) states, "attributes that have been estimated from the data must always have a name ending with trailing underscore, for example the coefficients of some regression estimator would be stored in a `coef_` attribute after `fit` has been called." In summary, the trailing underscore in class attributes is a scikit-learn's convention to denote that the attributes are estimated, or to denote that they are fitted attributes.

In [0]:
print("Assumption parameter (no underscore):", dev.average)
print("Estimated parameter (underscore):\n", dev.ldf_)

## Development Averaging

Now that we have a grounding in triangle manipulation and the basics of estimators, we can start getting more creative with customizing our development factors.

The basic `Development` estimator uses a weighted regression through the origin for estimating parameters. Mack showed that using weighted regressions allows for:
1. `volume` weighted average development patterns<br>
2. `simple` average development factors<br>
3. OLS `regression` estimate of development factor where the regression equation is Y = mX + 0<br>

While he posited this framework to suggest the `MackChainladder` stochastic method, it is an elegant form even for deterministic development pattern selection.

In [0]:
genins = cl.load_sample("genins")
genins

We can also print the `age_to_age` factors.

In [0]:
genins.age_to_age

And colorcode with `heatmap()`.

In [0]:
genins.age_to_age.heatmap()

In [0]:
vol = cl.Development(average="volume").fit(genins).ldf_
vol

In [0]:
sim = cl.Development(average="simple").fit(genins).ldf_
sim

In most cases, estimator attributes are `Triangle`s themselves and can be manipulated with just like raw triangles.

In [0]:
print("LDF Type: ", type(vol))
print("Difference between volume and simple average:")
vol - sim

We can specify how the LDFs are averaged independently for each age-to-age period. For example, we can use `volume` averaging on the first pattern, `simple` the second, `regression` the third, and then repeat the cycle three times for the 9 age-to-age factors that we need. Note that the array of selected method must be of the same length as the number of age-to-age factors.

In [0]:
cl.Development(average=["volume", "simple", "regression"] * 3).fit(genins).ldf_

Another example, using `volume`-weighting for the first factor, `simple`-weighting for the next 5 factors, and `volume`-weighting for the last 3 factors.

In [0]:
cl.Development(average=["volume"] + ["simple"] * 5 + ["volume"] * 3).fit(genins).ldf_

## Averaging Period

`Development` comes with an `n_periods` parameter that allows you to select the latest `n` origin periods for fitting your development patterns. `n_periods=-1` is used to indicate the usage of all available periods, which is also the default if the parameter is not specified. The units of `n_periods` follows the `origin_grain` of the underlying triangle.

In [0]:
cl.Development().fit(genins).ldf_

In [0]:
cl.Development(n_periods=-1).fit(genins).ldf_

In [0]:
cl.Development(n_periods=3).fit(genins).ldf_

Much like `average`, `n_periods` can also be set for each age-to-age individually.

In [0]:
cl.Development(n_periods=[8, 2, 6, 5, -1, 2, -1, -1, 5]).fit(genins).ldf_

Note that if we provide `n_periods` that is greater than what is available for any particular age-to-age period, all available periods will be used instead.

In [0]:
cl.Development(n_periods=[1, 2, 3, 4, 5, 6, 7, 8, 9]).fit(
    genins
).ldf_ == cl.Development(n_periods=[1, 2, 3, 4, 5, 4, 3, 2, 1]).fit(genins).ldf_

## Discarding Problematic Link Ratios

Even with `n_periods`, there are situations where you might want to be more surgical in our selections. For example, you could have a valuation period with bad data and wish to omit the entire diagonal from your averaging.

In [0]:
cl.Development(drop_valuation="2004").fit(genins).ldf_

We can also do an olympic averaging (i.e. exluding high and low from each period).

In [0]:
cl.Development(drop_high=True, drop_low=True).fit(genins).ldf_

The function also accepts intergers. For example, if we want to drop the highest 3 factors from each period.

In [0]:
cl.Development(drop_high=3).fit(genins).ldf_

There's a `preserve` that we can use, this variable allows us to specified the minimum number of LDFs required for calculation. If this minimum is not yet, the `drop_high` and `drop_low` for that age will be ignored. This is especially useful in the tail when the data is thin.

In [0]:
cl.Development(drop_high=3, drop_low=2, preserve=2).fit(genins).ldf_

We can also use an array of booleans or ints.

In [0]:
cl.Development(drop_high=[True, True, False, True], drop_low=[1, 2, 0, 3]).fit(
    genins
).ldf_

Or maybe there is just a single outlier link-ratio that you don't think is indicative of future development.  For these, you can specify the intersection of the origin and development age of the **denominator** of the link-ratio to `drop`.

In [0]:
genins.age_to_age.heatmap()

Let's say we believe the 4.5680 factor from origin 2004 between age 12 and 24 should be dropped, we can use `drop=('2004', 12)`.

In [0]:
cl.Development(drop=("2004", 12)).fit(genins).ldf_

If there are more than one outliers, you can also pass an array of array to the `drop` argument.

In [0]:
cl.Development(drop=[("2004", 12), ("2008", 24)]).fit(genins).ldf_

## Transformers
In `sklearn`, there are two types of estimators: transformers and predictors. A transformer transforms the input data (X) in some ways, and a predictor predicts a new value (or values, Y) by using the input data X.

`Development` is a transformer, as the returned object is a means to create development patterns, which is used to estimate ultimates, but itself is not a reserving model (predictor). 

Transformers come with the `tranform` and `fit_transform` method. These will return a `Triangle` object, but augment it with additional information for use in a subsequent IBNR model (a predictor). `drop_high` (and `drop_low`) can take an array of boolean variables, indicating if the highest factor should be dropped for each of the LDF calculation.

In [0]:
transformed_triangle = cl.Development(drop_high=[True] * 4 + [False] * 5).fit_transform(
    genins
)
transformed_triangle

In [0]:
transformed_triangle.link_ratio

Our transformed triangle behaves as our original `genins` triangle.  However, notice the link_ratios exclude any droppped values you specified.

In [0]:
transformed_triangle.link_ratio.heatmap()

In [0]:
print(type(transformed_triangle))
transformed_triangle.latest_diagonal

However, it has other attributes that make it IBNR model-ready.

In [0]:
transformed_triangle.cdf_

`fit_transform()` is equivalent to calling `fit` and `transform` in succession on the same triangle.  Again, this should feel very familiar to the `sklearn` practitioner.

In [0]:
cl.Development().fit_transform(genins) == cl.Development().fit(genins).transform(genins)

The reason you might want want to use `fit` and `transform` separately would be when you want to apply development patterns to a a different triangle.  For example, we can:

1. Extract the commercial auto triangles from the `clrd` dataset<br>
2. Summarize to an industry level and `fit` a `Development` object<br>
3. We can then `transform` the individual company triangles with the industry development patterns<br>

In [0]:
clrd = cl.load_sample("clrd")
comauto = clrd[clrd["LOB"] == "comauto"]["CumPaidLoss"]

comauto_industry = comauto.sum()
industry_dev = cl.Development().fit(comauto_industry)

industry_dev.transform(comauto)

## Working with Multidimensional Triangles

Several (though not all) of the estimators in `chainladder` can be fit to several triangles simultaneously. While this can be a convenient shorthand, all these estimators use the same assumptions across every triangle.

In [0]:
clrd = cl.load_sample("clrd").groupby("LOB").sum()["CumPaidLoss"]
print("Fitting to " + str(len(clrd.index)) + " industries simultaneously.")
cl.Development().fit_transform(clrd).cdf_

For greater control, you can slice individual triangles out and fit separate patterns to each.

In [0]:
print(cl.Development(average="simple").fit(clrd.loc["wkcomp"]))
print(cl.Development(n_periods=4).fit(clrd.loc["ppauto"]))
print(cl.Development(average="regression", n_periods=6).fit(clrd.loc["comauto"]))

In [0]:
def test_func():
    """
    This is a test:
    >>> 2 + 2
    5
    """
    return None